1. Loading Dependencies

In [18]:
# ============================================================
# DADDA Experiments Notebook
# Section 1 — Imports
# ============================================================

import os
import sys

sys.path.append("..")

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim

from src.data.preprocessing import (
    load_image_paths,
    build_dataset,
    create_labels,
    split_dataset,
)

from src.data.dataset import ADDADataset

from src.data.loaders import (
    create_source_loader,
    create_target_loader,
)

print("Imports completed.")

Imports completed.


2. Experiment Configuration

In [19]:
IMAGE_SIZE = (120, 120)

BATCH_SIZE = 256

NUM_CLASSES = 3

SEED = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

# ============================================================
# Hyperparameters
# ============================================================

batch_size = 256

criterion_task = nn.CrossEntropyLoss()

criterion_disc = nn.BCEWithLogitsLoss()

alpha = 1

EPS = 1e-8

epochs = 150

lr = 0.0001

Device: cuda


3. Load Source and Target Domain Data

In [3]:
# ============================================================
# Source Domain Directories
# ============================================================

SOURCE_ROOT = "../data/source"

SOURCE_NORMAL_DIR = os.path.join(
    SOURCE_ROOT,
    "normal"
)

SOURCE_PLUME_DIR = os.path.join(
    SOURCE_ROOT,
    "plume"
)

SOURCE_SPATTER_DIR = os.path.join(
    SOURCE_ROOT,
    "spatter"
)

# ============================================================
# Target Domain Directories
# ============================================================

TARGET_ROOT = "../data/target"

TARGET_NORMAL_DIR = os.path.join(
    TARGET_ROOT,
    "normal"
)

TARGET_PLUME_DIR = os.path.join(
    TARGET_ROOT,
    "plume"
)

TARGET_SPATTER_DIR = os.path.join(
    TARGET_ROOT,
    "spatter"
)

In [4]:
# ============================================================
# Load Source Image Paths
# ============================================================

source_normal_paths = load_image_paths(
    SOURCE_NORMAL_DIR,
    extension="png"
)

source_plume_paths = load_image_paths(
    SOURCE_PLUME_DIR,
    extension="png"
)

source_spatter_paths = load_image_paths(
    SOURCE_SPATTER_DIR,
    extension="png"
)

# ============================================================
# Load Target Image Paths
# ============================================================

target_normal_paths = load_image_paths(
    TARGET_NORMAL_DIR,
    extension="png"
)

target_plume_paths = load_image_paths(
    TARGET_PLUME_DIR,
    extension="png"
)

target_spatter_paths = load_image_paths(
    TARGET_SPATTER_DIR,
    extension="png"
)

print("Source normal:", len(source_normal_paths))
print("Source plume:", len(source_plume_paths))
print("Source spatter:", len(source_spatter_paths))

print("Target normal:", len(target_normal_paths))
print("Target plume:", len(target_plume_paths))
print("Target spatter:", len(target_spatter_paths))

Source normal: 1000
Source plume: 999
Source spatter: 1000
Target normal: 320
Target plume: 292
Target spatter: 386


In [5]:
# ============================================================
# Build Source Datasets
# ============================================================

source_normal_images = build_dataset(
    image_paths=source_normal_paths,
    image_size=IMAGE_SIZE,
    grayscale=True,
    gaussian_blur=False,
)

source_plume_images = build_dataset(
    image_paths=source_plume_paths,
    image_size=IMAGE_SIZE,
    grayscale=True,
    gaussian_blur=False,
)

source_spatter_images = build_dataset(
    image_paths=source_spatter_paths,
    image_size=IMAGE_SIZE,
    grayscale=True,
    gaussian_blur=False,
)

# ============================================================
# Build Target Datasets
# ============================================================

target_normal_images = build_dataset(
    image_paths=target_normal_paths,
    image_size=IMAGE_SIZE,
    grayscale=True,
    gaussian_blur=False,
)

target_plume_images = build_dataset(
    image_paths=target_plume_paths,
    image_size=IMAGE_SIZE,
    grayscale=True,
    gaussian_blur=False,
)

target_spatter_images = build_dataset(
    image_paths=target_spatter_paths,
    image_size=IMAGE_SIZE,
    grayscale=True,
    gaussian_blur=False,
)

print("Datasets loaded.")

Datasets loaded.


In [6]:
# ============================================================
# Create Source Labels
# ============================================================

source_normal_labels = create_labels(
    len(source_normal_images),
    class_id=0
)

source_plume_labels = create_labels(
    len(source_plume_images),
    class_id=1
)

source_spatter_labels = create_labels(
    len(source_spatter_images),
    class_id=2
)

# ============================================================
# Create Target Labels
# ============================================================

target_normal_labels = create_labels(
    len(target_normal_images),
    class_id=0
)

target_plume_labels = create_labels(
    len(target_plume_images),
    class_id=1
)

target_spatter_labels = create_labels(
    len(target_spatter_images),
    class_id=2
)

In [7]:
# ============================================================
# Combine Source Dataset
# ============================================================

X_source = np.concatenate([
    source_normal_images,
    source_plume_images,
    source_spatter_images,
], axis=0)

y_source = np.concatenate([
    source_normal_labels,
    source_plume_labels,
    source_spatter_labels,
], axis=0)

# ============================================================
# Combine Target Dataset
# ============================================================

X_target = np.concatenate([
    target_normal_images,
    target_plume_images,
    target_spatter_images,
], axis=0)

y_target = np.concatenate([
    target_normal_labels,
    target_plume_labels,
    target_spatter_labels,
], axis=0)

print("Source dataset shape:", X_source.shape)
print("Target dataset shape:", X_target.shape)

Source dataset shape: (2999, 120, 120, 1)
Target dataset shape: (998, 120, 120, 1)


In [8]:
# ============================================================
# Split Source Dataset
# ============================================================

(
    X_source_train,
    y_source_train,
    X_source_val,
    y_source_val,
    X_source_test,
    y_source_test,
) = split_dataset(
    X_source,
    y_source,
    train_ratio=0.7,
    val_ratio=0.15,
    seed=SEED,
)

# ============================================================
# Split Target Dataset
# ============================================================

(
    X_target_train,
    y_target_train,
    X_target_val,
    y_target_val,
    X_target_test,
    y_target_test,
) = split_dataset(
    X_target,
    y_target,
    train_ratio=0.7,
    val_ratio=0.15,
    seed=SEED,
)

print("Source train:", X_source_train.shape)
print("Target train:", X_target_train.shape)

Source train: (2099, 120, 120, 1)
Target train: (698, 120, 120, 1)


In [9]:
# ============================================================
# Create Source Dataset Objects
# ============================================================

source_train_dataset = ADDADataset(
    X_source_train,
    y_source_train
)

source_val_dataset = ADDADataset(
    X_source_val,
    y_source_val
)

source_test_dataset = ADDADataset(
    X_source_test,
    y_source_test
)

# ============================================================
# Create Target Dataset Objects
# ============================================================

target_train_dataset = ADDADataset(
    X_target_train,
    y_target_train
)

target_val_dataset = ADDADataset(
    X_target_val,
    y_target_val
)

target_test_dataset = ADDADataset(
    X_target_test,
    y_target_test
)

print("Dataset objects created.")

Dataset objects created.


In [10]:
# ============================================================
# Create Source DataLoaders
# ============================================================

source_train_loader = create_source_loader(
    source_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

source_val_loader = create_source_loader(
    source_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

source_test_loader = create_source_loader(
    source_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

# ============================================================
# Create Target DataLoaders
# ============================================================

target_train_loader = create_target_loader(
    target_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
)

target_val_loader = create_target_loader(
    target_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

target_test_loader = create_target_loader(
    target_test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
)

print("All dataloaders created.")

All dataloaders created.


In [11]:
# ============================================================
# Batch Shape Check
# ============================================================

source_images, source_labels = next(
    iter(source_train_loader)
)

target_images, target_labels = next(
    iter(target_train_loader)
)

print("Source batch images:", source_images.shape)
print("Source batch labels:", source_labels.shape)

print("Target batch images:", target_images.shape)
print("Target batch labels:", target_labels.shape)

print("Data pipeline completed.")

Source batch images: torch.Size([256, 1, 120, 120])
Source batch labels: torch.Size([256])
Target batch images: torch.Size([256, 1, 120, 120])
Target batch labels: torch.Size([256])
Data pipeline completed.


4. Train DADDA Model

In [12]:
from src.models.dadda import DADDA

from src.training.train_dadda import (
    train_dadda,
)

In [20]:
# ============================================================
# Instantiate Model
# ============================================================

model = DADDA(
    # embedding_dim=20,
    # num_classes=NUM_CLASSES,
)

print(model)

DADDA(
  (encoder): ConvolutionalEncoder(
    (feature_extractor): Sequential(
      (0): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (4): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (5): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (8): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (9): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (10): ReLU()
      (11): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
      (12): Conv2d(32, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
      (13): BatchNorm2d(16

In [24]:
# ============================================================
# Create Optimizers
# ============================================================

optimizer_encoder = optim.Adam(
    model.encoder.parameters(),
    lr=lr,
)

optimizer_task_classifiers = optim.Adam(

    list(model.task_classifier_1.parameters())
    +
    list(model.task_classifier_2.parameters()),

    lr=lr,
)

optimizer_domain_classifier = optim.Adam(
    model.domain_classifier.parameters(),
    lr=lr,
)

In [25]:
# ============================================================
# Launch Training
# ============================================================

history = train_dadda(

    model=model,

    source_loader=source_train_loader,

    target_loader=target_train_loader,

    optimizer_encoder=optimizer_encoder,

    optimizer_task_classifiers=(
        optimizer_task_classifiers
    ),

    optimizer_domain_classifier=(
        optimizer_domain_classifier
    ),

    classification_criterion=(
        criterion_task
    ),

    device=DEVICE,

    num_epochs=epochs,

    lambda_discrepancy=1.0,

    lambda_domain=alpha,
)

Epoch 1/150: 100%|██████████| 9/9 [00:00<00:00, 13.13it/s, classification=2.01, discrepancy=0.0379, domain=1.38, total=3.43]


Epoch [1/150] Classification: 2.0857 | Discrepancy: 0.0488 | Domain: 1.3842 | Total: 3.5186


Epoch 2/150: 100%|██████████| 9/9 [00:00<00:00, 23.12it/s, classification=1.76, discrepancy=0.0294, domain=1.38, total=3.18]


Epoch [2/150] Classification: 1.8693 | Discrepancy: 0.0314 | Domain: 1.3831 | Total: 3.2838


Epoch 3/150: 100%|██████████| 9/9 [00:00<00:00, 23.12it/s, classification=1.82, discrepancy=0.0239, domain=1.38, total=3.22]


...

Epoch [148/150] Classification: 0.0586 | Discrepancy: 0.0013 | Domain: 0.0037 | Total: 0.0636


Epoch 149/150: 100%|██████████| 9/9 [00:00<00:00, 23.33it/s, classification=0.0812, discrepancy=0.000973, domain=0.00386, total=0.086] 


Epoch [149/150] Classification: 0.0549 | Discrepancy: 0.0014 | Domain: 0.0037 | Total: 0.0599


Epoch 150/150: 100%|██████████| 9/9 [00:00<00:00, 23.26it/s, classification=0.0533, discrepancy=0.00244, domain=0.00415, total=0.0599] 

Epoch [150/150] Classification: 0.0497 | Discrepancy: 0.0013 | Domain: 0.0036 | Total: 0.0546
